# Notebook 4: Data Warehouse & Power BI Reporting
**H9DISS1 Data Intensive Scalable Systems | NCI MSc Data Analytics 2025/26**

---

This notebook:
1. **Promotes Gold Delta tables → Fabric Data Warehouse** using T-SQL DDL + COPY
2. **Runs analytical SQL queries** directly on the Warehouse for ad-hoc analysis
3. **Creates Power BI semantic model** by publishing the Warehouse as a dataset
4. **Documents Power BI DAX measures** for the dashboard

### Fabric Data Flow — Final Layer
```
Lakehouse Tables/ (Gold Delta)
        ↓  Notebook 4 copies to
Fabric Data Warehouse (T-SQL)
        ↓  Power BI connects via
Semantic Model → Dashboard → Report
```

### Why a Separate Data Warehouse?
The Lakehouse Delta tables are optimised for Spark batch reads.
The Fabric Data Warehouse adds a **T-SQL endpoint** — enabling:
- Direct SQL queries from analysts without Spark
- Power BI DirectQuery mode (live connection, not cached)
- Row-level security for multi-user dashboards
- Standard ANSI SQL for cross-tool compatibility

## Cell 1: Copy Gold Tables into Fabric Warehouse via Spark JDBC

In Microsoft Fabric, the Warehouse SQL endpoint is exposed as a JDBC connection.
We use Spark's `.write.jdbc()` to push the Gold DataFrames directly.

In [ ]:
# ── Fabric Warehouse JDBC connection ──────────────────────────────────────
# Replace <WORKSPACE_ID> and <WAREHOUSE_ID> with your actual values from
# Fabric Portal → Warehouse → Settings → SQL connection string

WORKSPACE_ID  = "<YOUR_WORKSPACE_ID>"
WAREHOUSE_ID  = "<YOUR_WAREHOUSE_ID>"
WAREHOUSE_URL = (
    f"jdbc:sqlserver://{WORKSPACE_ID}.datawarehouse.fabric.microsoft.com:1433;"
    f"database={WAREHOUSE_ID};encrypt=true;trustServerCertificate=false;"
    f"hostNameInCertificate=*.datawarehouse.fabric.microsoft.com;"
    f"loginTimeout=30;authentication=ActiveDirectoryServicePrincipal"
)

# Use Service Principal credentials (set in Fabric Key Vault or environment)
import os
SP_CLIENT_ID     = os.environ.get("FABRIC_SP_CLIENT_ID", "<client_id>")
SP_CLIENT_SECRET = os.environ.get("FABRIC_SP_SECRET",    "<client_secret>")

jdbc_props = {
    "driver":         "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    "user":           SP_CLIENT_ID,
    "password":       SP_CLIENT_SECRET,
    "batchsize":      "10000",
    "isolationLevel": "NONE",
}

# Gold tables to promote to Warehouse
GOLD_TABLES = [
    "gold_analysis1_eq_aid_correlation",
    "gold_analysis2_multihazard_profile",
    "gold_analysis2_eq_state_profile",
    "gold_analysis3_national_trend",
    "gold_analysis3_annual_by_state",
    "gold_summary_statistics",
]

for tbl in GOLD_TABLES:
    df = spark.table(tbl)
    df.write \
        .mode("overwrite") \
        .jdbc(url=WAREHOUSE_URL, table=f"dbo.{tbl}", properties=jdbc_props)
    print(f"  ✓ Copied {tbl} → Warehouse dbo.{tbl} ({df.count():,} rows)")

print("\nAll Gold tables promoted to Fabric Data Warehouse.")

## Cell 2: Analytical SQL Queries via Spark SQL

These queries can also be run directly in the Fabric Warehouse Query Editor
using T-SQL. They are shown here as Spark SQL for notebook reproducibility.

In [ ]:
# ── Query 1: States ranked by Seismic Risk Index ───────────────────────────
print("── Query 1: State Seismic Risk Ranking ──────────────────")
spark.sql("""
    SELECT
        state,
        ROUND(sri, 4)                        AS seismic_risk_index,
        county_declarations,
        ia_activations,
        ROUND(ia_rate * 100, 1)              AS ia_rate_pct,
        ROUND(pa_rate * 100, 1)              AS pa_rate_pct,
        CASE
            WHEN ia_rate = 0   THEN 'No Residential Coverage'
            WHEN ia_rate < 0.5 THEN 'Low Coverage'
            WHEN ia_rate < 0.8 THEN 'Moderate Coverage'
            ELSE                    'High Coverage'
        END                                  AS coverage_tier
    FROM gold_analysis1_eq_aid_correlation
    ORDER BY seismic_risk_index DESC
""").show(14, truncate=False)

In [ ]:
# ── Query 2: Insurance gap analysis — states with high PA but low IA ───────
print("── Query 2: Insurance Gap — PA vs IA Divergence ─────────")
spark.sql("""
    SELECT
        state,
        county_declarations,
        ia_activations,
        pa_activations,
        ROUND((pa_rate - ia_rate) * 100, 1)  AS ia_pa_gap_pct,
        ROUND(ia_rate  * 100, 1)             AS ia_pct,
        ROUND(pa_rate  * 100, 1)             AS pa_pct,
        ROUND(sri, 4)                        AS sri
    FROM gold_analysis1_eq_aid_correlation
    WHERE pa_rate > ia_rate
    ORDER BY ia_pa_gap_pct DESC
""").show(truncate=False)

In [ ]:
# ── Query 3: Temporal shift — IA share pre vs post 2010 ───────────────────
print("── Query 3: Pre/Post 2010 Insurance Gap ─────────────────")
spark.sql("""
    SELECT
        CASE WHEN year < 2010 THEN 'Pre-2010'
             ELSE 'Post-2010' END          AS period,
        COUNT(*)                           AS years_count,
        SUM(total_county_rows)             AS total_county_rows,
        SUM(total_ia)                      AS total_ia_activations,
        SUM(total_pa)                      AS total_pa_activations,
        ROUND(
            SUM(total_ia)
            / NULLIF(SUM(total_county_rows), 0) * 100, 1
        )                                  AS ia_share_pct,
        ROUND(
            SUM(total_pa)
            / NULLIF(SUM(total_county_rows), 0) * 100, 1
        )                                  AS pa_share_pct
    FROM gold_analysis3_national_trend
    WHERE total_county_rows IS NOT NULL
    GROUP BY period
    ORDER BY period DESC
""").show(truncate=False)

In [ ]:
# ── Query 4: Multi-hazard earthquake share vs IA activation ────────────────
print("── Query 4: Earthquake vs Other Hazards in Same States ──")
spark.sql("""
    SELECT
        e.state,
        e.county_rows                      AS eq_county_rows,
        e.ia_activations                   AS eq_ia_activations,
        ROUND(e.ia_rate * 100, 1)          AS eq_ia_pct,
        e.disaster_share_pct               AS eq_share_of_total,
        e.state_total                      AS total_all_disasters,
        ROUND(a1.sri, 4)                   AS sri
    FROM gold_analysis2_eq_state_profile  e
    JOIN gold_analysis1_eq_aid_correlation a1 ON e.state = a1.state
    ORDER BY eq_ia_pct DESC
""").show(truncate=False)

## Cell 3: Power BI — DAX Measures for Dashboard

After the Warehouse tables are created, connect Power BI:
1. Open Power BI Desktop
2. **Get Data → Microsoft Fabric Data Warehouse**
3. Select your Warehouse and the six `dbo.gold_*` tables
4. Add the DAX measures below to the semantic model

The cell below documents all required DAX measures as strings
(for copy-paste into Power BI Desktop measure editor).

In [ ]:
dax_measures = [
    {
        "name":  "Total EQ County Declarations",
        "table": "gold_analysis1_eq_aid_correlation",
        "dax":   "Total EQ County Declarations = SUM(gold_analysis1_eq_aid_correlation[county_declarations])"
    },
    {
        "name":  "Total IA Activations",
        "table": "gold_analysis1_eq_aid_correlation",
        "dax":   "Total IA Activations = SUM(gold_analysis1_eq_aid_correlation[ia_activations])"
    },
    {
        "name":  "National IA Rate",
        "table": "gold_analysis1_eq_aid_correlation",
        "dax":   """
National IA Rate =
DIVIDE(
    SUM(gold_analysis1_eq_aid_correlation[ia_activations]),
    SUM(gold_analysis1_eq_aid_correlation[county_declarations]),
    0
)"""
    },
    {
        "name":  "Insurance Gap Score",
        "table": "gold_analysis1_eq_aid_correlation",
        "dax":   """
Insurance Gap Score =
AVERAGEX(
    gold_analysis1_eq_aid_correlation,
    gold_analysis1_eq_aid_correlation[pa_rate]
    - gold_analysis1_eq_aid_correlation[ia_rate]
)"""
    },
    {
        "name":  "Peak Declaration Year",
        "table": "gold_analysis3_national_trend",
        "dax":   """
Peak Declaration Year =
MAXX(
    TOPN(1,
        gold_analysis3_national_trend,
        gold_analysis3_national_trend[total_county_rows],
        DESC
    ),
    gold_analysis3_national_trend[year]
)"""
    },
    {
        "name":  "Post-2010 IA Share",
        "table": "gold_analysis3_national_trend",
        "dax":   """
Post-2010 IA Share =
DIVIDE(
    CALCULATE(
        SUM(gold_analysis3_national_trend[total_ia]),
        gold_analysis3_national_trend[year] >= 2010
    ),
    CALCULATE(
        SUM(gold_analysis3_national_trend[total_county_rows]),
        gold_analysis3_national_trend[year] >= 2010
    ),
    0
)"""
    },
    {
        "name":  "Coverage Tier Label",
        "table": "gold_analysis1_eq_aid_correlation",
        "dax":   """
Coverage Tier Label =
SWITCH(
    TRUE(),
    gold_analysis1_eq_aid_correlation[ia_rate] = 0,     "No Residential Coverage",
    gold_analysis1_eq_aid_correlation[ia_rate] < 0.5,   "Low Coverage (<50%)",
    gold_analysis1_eq_aid_correlation[ia_rate] < 0.8,   "Moderate Coverage (50-80%)",
    "High Coverage (>80%)"
)"""
    },
]

print("Power BI DAX Measures — copy into Power BI Desktop semantic model:\n")
for m in dax_measures:
    print(f"{'='*60}")
    print(f"Measure: {m['name']}")
    print(f"Table:   {m['table']}")
    print(f"DAX:\n{m['dax'].strip()}")
    print()

## Cell 4: Recommended Power BI Dashboard Layout

Create a 4-page Power BI report using the Warehouse tables:

In [ ]:
dashboard_spec = {
    "Page 1 — Executive Summary": [
        "Card: Total EQ County Declarations (228)",
        "Card: National IA Rate (30.3%)",
        "Card: Insurance Gap Score (avg PA-IA gap)",
        "Card: Peak Declaration Year (2020)",
        "Map: State-level SRI choropleth (size = county_declarations)",
        "Bar chart: Top 10 states by SRI",
    ],
    "Page 2 — State Analysis (RQ1 + RQ2)": [
        "Scatter: SRI vs IA Rate (from gold_analysis1, with state labels)",
        "Stacked bar: IA vs PA activations by state",
        "Table: Full state table with Coverage Tier Label slicer",
        "Slicer: Filter by coverage_tier",
    ],
    "Page 3 — Multi-Hazard Profile (RQ2)": [
        "Clustered bar: Declarations by incidentType for seismic states",
        "Matrix: IA rate by state × incidentType (gold_analysis2_multihazard)",
        "Bubble: county_rows vs ia_activations (size=total_declarations, colour=sri)",
        "Callout card: Puerto Rico anomaly — 112 county rows, 0 IA",
    ],
    "Page 4 — Temporal Trends (RQ3)": [
        "Line + area chart: total_county_rows by year (gold_analysis3_national_trend)",
        "Stacked bar: total_ia + total_pa by year",
        "Reference line: 2010 threshold",
        "KPI card: Post-2010 IA Share DAX measure vs 88% pre-2010 benchmark",
    ],
}

for page, visuals in dashboard_spec.items():
    print(f"\n{'='*55}")
    print(f"  {page}")
    print(f"{'='*55}")
    for v in visuals:
        print(f"  • {v}")

## Cell 5: Final Pipeline Summary

In [ ]:
print("="*65)
print("  FULL FABRIC PIPELINE — COMPLETE")
print("="*65)
print()
print("  LAKEHOUSE — Medallion Architecture")
print("  ├── Files/ (Bronze)")
print("  │   └── earthquake_insurance/raw/DisasterDeclarationsSummaries.csv")
print("  └── Tables/ (Silver + Gold Delta)")
print("      ├── silver_fema_all_disasters        (69,896 rows)")
print("      ├── silver_usgs_earthquakes          (~20,000 rows)")
print("      ├── silver_seismic_risk_index        (14 states)")
print("      ├── gold_analysis1_eq_aid_correlation (14 states)")
print("      ├── gold_analysis2_multihazard_profile (107 combos)")
print("      ├── gold_analysis2_eq_state_profile  (11 states)")
print("      ├── gold_analysis3_national_trend    (13 years)")
print("      ├── gold_analysis3_annual_by_state   (state×year)")
print("      └── gold_summary_statistics          (13 metrics)")
print()
print("  DATA WAREHOUSE — T-SQL endpoint")
print("  └── dbo.gold_* (6 tables, T-SQL queryable)")
print()
print("  POWER BI — Semantic model")
print("  └── 4-page dashboard (DirectQuery from Warehouse)")
print()
print("="*65)